In [16]:
import json, nbformat

In [ ]:
!pip install transformers accelerate bitsandbytes -q

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import json
import re
import time
import torch
from collections import deque
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-8B"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True,
    trust_remote_code=True,
)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
model.eval()
print("Modelo cargado")

In [ ]:
def qwen_batch(prompts: list,
               system: str | None = None,
               max_new_tokens: int = 512,
               enable_thinking: bool = False) -> list:
    texts = []
    for prompt in prompts:
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        text = tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=False,
            enable_thinking=enable_thinking,
        )
        texts.append(text)

    inputs = tokenizer(texts, return_tensors="pt", padding=True).to(model.device)

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    with torch.no_grad():
        out = model.generate(**inputs, **gen_kwargs)

    input_len = inputs["input_ids"].shape[1]
    results = []
    for i in range(len(prompts)):
        gen_ids = out[i, input_len:]
        results.append(tokenizer.decode(gen_ids, skip_special_tokens=True))
    return results

In [ ]:
def is_block_domain(scenario_context: str) -> bool:
    ctx = scenario_context.lower()
    return "block" in ctx and "mount_node" in ctx


def strip_decorations(line: str) -> str:
    line = re.sub(r"^[\-\*\u2022]\s*", "", line)
    line = re.sub(r"^\d+[\.\.)]\s*", "", line)
    line = line.replace("**", "").replace("`", "")
    return line.strip()


def parse_block_line(line: str):
    m = re.match(r"pick\s+up\s+(?:the\s+)?(\w+)(?:\s+block)?", line, re.I)
    if m:
        return f"(engage_payload {m.group(1).lower()})"
    m = re.match(r"put\s+down\s+(?:the\s+)?(\w+)(?:\s+block)?", line, re.I)
    if m:
        return f"(release_payload {m.group(1).lower()})"
    m = re.match(r"(?:unmount_node|unstack)\s+(?:the\s+)?(\w+)(?:\s+block)?\s+from\s+(?:on\s+top\s+of\s+)?(?:the\s+)?(\w+)", line, re.I)
    if m:
        return f"(unmount_node {m.group(1).lower()} {m.group(2).lower()})"
    m = re.match(r"(?:mount_node|stack)\s+(?:the\s+)?(\w+)(?:\s+block)?\s+on\s+top\s+of\s+(?:the\s+)?(\w+)", line, re.I)
    if m:
        return f"(mount_node {m.group(1).lower()} {m.group(2).lower()})"
    m = re.match(r"engage_payload\s+(?:the\s+)?(\w+)", line, re.I)
    if m:
        return f"(engage_payload {m.group(1).lower()})"
    m = re.match(r"release_payload\s+(?:the\s+)?(\w+)", line, re.I)
    if m:
        return f"(release_payload {m.group(1).lower()})"
    return None


def parse_feast_line(line: str):
    m = re.match(r"(attack|succumb)\s+object\s+(\w+)", line, re.I)
    if m:
        return f"({m.group(1).lower()} {m.group(2).lower()})"
    m = re.match(r"(feast|overcome)\s+object\s+(\w+)\s+from\s+object\s+(\w+)", line, re.I)
    if m:
        return f"({m.group(1).lower()} {m.group(2).lower()} {m.group(3).lower()})"
    return None


def parse_plan(generated_text: str, is_block: bool) -> list:
    text = generated_text.split("[PLAN END]")[0]
    text = text.split("[STATEMENT]")[0]
    lines = [l.strip().rstrip(".") for l in text.splitlines() if l.strip()]
    actions = []
    for raw_line in lines:
        line = strip_decorations(raw_line)
        if not line:
            continue
        action = parse_block_line(line) if is_block else parse_feast_line(line)
        if action:
            actions.append(action)
    return actions

In [ ]:
FEAST_RULES = {
    "attack": {"pre": [("province", 1), ("planet", 1), ("harmony", 0)],
               "add": [("pain", 1)],
               "del": [("province", 1), ("planet", 1), ("harmony", 0)]},
    "succumb": {"pre": [("pain", 1)],
                "add": [("province", 1), ("planet", 1), ("harmony", 0)],
                "del": [("pain", 1)]},
    "overcome": {"pre": [("province", 2), ("pain", 1)],
                 "add": [("harmony", 0), ("province", 1), ("craves", (1, 2))],
                 "del": [("province", 2), ("pain", 1)]},
    "feast": {"pre": [("craves", (1, 2)), ("province", 1), ("harmony", 0)],
              "add": [("pain", 1), ("province", 2)],
              "del": [("craves", (1, 2)), ("province", 1), ("harmony", 0)]},
}

BLOCK_RULES = {
    "engage_payload": {"pre": [("ontable", 1), ("clear", 1), ("handempty", 0)],
                        "add": [("holding", 1)],
                        "del": [("ontable", 1), ("clear", 1), ("handempty", 0)]},
    "release_payload": {"pre": [("holding", 1)],
                         "add": [("ontable", 1), ("clear", 1), ("handempty", 0)],
                         "del": [("holding", 1)]},
    "mount_node": {"pre": [("holding", 1), ("clear", 2)],
                   "add": [("on", (1, 2)), ("clear", 1), ("handempty", 0)],
                   "del": [("holding", 1), ("clear", 2)]},
    "unmount_node": {"pre": [("on", (1, 2)), ("clear", 1), ("handempty", 0)],
                      "add": [("holding", 1), ("clear", 2)],
                      "del": [("on", (1, 2)), ("clear", 1), ("handempty", 0)]},
}

In [ ]:
def parse_feast_state(scenario_context: str):
    stmt_blocks = scenario_context.split("[STATEMENT]")
    last = stmt_blocks[-1]
    init_match = re.search(r"As initial conditions I have that,(.+?)\.\s*\n", last, re.S)
    goal_match = re.search(r"My goal is to have that(.+?)\.\s*\n", last, re.S)

    facts = set()
    if init_match:
        chunk = init_match.group(1)
        chunk = chunk.replace(" and ", ", ")
        for part in chunk.split(","):
            part = part.strip()
            if not part:
                continue
            m = re.match(r"object (\w+) craves object (\w+)", part)
            if m:
                facts.add(("craves", m.group(1), m.group(2)))
                continue
            m = re.match(r"planet object (\w+)", part)
            if m:
                facts.add(("planet", m.group(1)))
                continue
            m = re.match(r"province object (\w+)", part)
            if m:
                facts.add(("province", m.group(1)))
                continue
            if part == "harmony":
                facts.add(("harmony",))

    goals = set()
    if goal_match:
        chunk = goal_match.group(1).replace(" and ", ", ")
        for part in chunk.split(","):
            part = part.strip()
            m = re.match(r"object (\w+) craves object (\w+)", part)
            if m:
                goals.add(("craves", m.group(1), m.group(2)))

    return facts, goals


def parse_block_state(scenario_context: str):
    stmt_blocks = scenario_context.split("[STATEMENT]")
    last = stmt_blocks[-1]
    init_match = re.search(r"As initial conditions I have that,(.+?)\.\s*\n", last, re.S)
    goal_match = re.search(r"My goal is to have that(.+?)\.\s*\n", last, re.S)

    facts = set()
    blocks = set()
    if init_match:
        chunk = init_match.group(1).replace(" and ", ", ")
        for part in chunk.split(","):
            part = part.strip()
            if not part:
                continue
            m = re.match(r"the (\w+) block is unobstructed", part)
            if m:
                facts.add(("clear", m.group(1)))
                blocks.add(m.group(1))
                continue
            if part == "the hand is empty":
                facts.add(("handempty",))
                continue
            m = re.match(r"the (\w+) block is on top of the (\w+) block", part)
            if m:
                facts.add(("on", m.group(1), m.group(2)))
                blocks.add(m.group(1))
                blocks.add(m.group(2))
                continue
            m = re.match(r"the (\w+) block is on the table", part)
            if m:
                facts.add(("ontable", m.group(1)))
                blocks.add(m.group(1))
                continue

    goals = set()
    if goal_match:
        chunk = goal_match.group(1).replace(" and ", ", ")
        for part in chunk.split(","):
            part = part.strip()
            m = re.match(r"the (\w+) block is on top of the (\w+) block", part)
            if m:
                goals.add(("on", m.group(1), m.group(2)))

    return facts, goals, blocks

In [ ]:
def fact_matches(fact_tuple, args, predicate, slot):
    if fact_tuple[0] != predicate:
        return False
    if isinstance(slot, tuple):
        a, b = slot
        return fact_tuple[1] == args.get(a) and fact_tuple[2] == args.get(b)
    else:
        return fact_tuple[1] == args.get(slot)


def check_pre(facts, rules, action, args):
    for predicate, slot in rules[action]["pre"]:
        if predicate in ("harmony", "handempty"):
            if (predicate,) not in facts:
                return False
            continue
        ok = False
        for f in facts:
            if fact_matches(f, args, predicate, slot):
                ok = True
                break
        if not ok:
            return False
    return True


def apply_effects(facts, rules, action, args):
    new_facts = set(facts)
    for predicate, slot in rules[action]["del"]:
        if predicate in ("harmony", "handempty"):
            new_facts.discard((predicate,))
            continue
        to_remove = None
        for f in new_facts:
            if fact_matches(f, args, predicate, slot):
                to_remove = f
                break
        if to_remove:
            new_facts.discard(to_remove)
    for predicate, slot in rules[action]["add"]:
        if predicate in ("harmony", "handempty"):
            new_facts.add((predicate,))
            continue
        if isinstance(slot, tuple):
            a, b = slot
            new_facts.add((predicate, args.get(a), args.get(b)))
        else:
            new_facts.add((predicate, args.get(slot)))
    return new_facts


def legal_actions_feast(facts, objects):
    legal = []
    for obj1 in objects:
        args = {1: obj1}
        if check_pre(facts, FEAST_RULES, "attack", args):
            legal.append(("attack", obj1, None))
        if check_pre(facts, FEAST_RULES, "succumb", args):
            legal.append(("succumb", obj1, None))
        for obj2 in objects:
            if obj1 == obj2:
                continue
            args2 = {1: obj1, 2: obj2}
            if check_pre(facts, FEAST_RULES, "overcome", args2):
                legal.append(("overcome", obj1, obj2))
            if check_pre(facts, FEAST_RULES, "feast", args2):
                legal.append(("feast", obj1, obj2))
    return legal


def legal_actions_block(facts, blocks):
    legal = []
    for b1 in blocks:
        args = {1: b1}
        if check_pre(facts, BLOCK_RULES, "engage_payload", args):
            legal.append(("engage_payload", b1, None))
        if check_pre(facts, BLOCK_RULES, "release_payload", args):
            legal.append(("release_payload", b1, None))
        for b2 in blocks:
            if b1 == b2:
                continue
            args2 = {1: b1, 2: b2}
            if check_pre(facts, BLOCK_RULES, "mount_node", args2):
                legal.append(("mount_node", b1, b2))
            if check_pre(facts, BLOCK_RULES, "unmount_node", args2):
                legal.append(("unmount_node", b1, b2))
    return legal


def action_to_tuple_str(action):
    verb, a1, a2 = action
    return f"({verb} {a1} {a2})" if a2 else f"({verb} {a1})"


def bfs_plan_feast(facts, goals, objects, max_depth=12):
    start = frozenset(facts)
    goal_set = frozenset(goals)
    if goal_set.issubset(start):
        return []
    visited = {start}
    queue = deque([(start, [])])
    while queue:
        cur_facts, path = queue.popleft()
        if len(path) >= max_depth:
            continue
        for action in legal_actions_feast(cur_facts, objects):
            verb, a1, a2 = action
            args = {1: a1, 2: a2} if a2 else {1: a1}
            new_facts = frozenset(apply_effects(cur_facts, FEAST_RULES, verb, args))
            new_path = path + [action]
            if goal_set.issubset(new_facts):
                return new_path
            if new_facts not in visited:
                visited.add(new_facts)
                queue.append((new_facts, new_path))
    return None


def bfs_plan_block(facts, goals, blocks, max_depth=16):
    start = frozenset(facts)
    goal_set = frozenset(goals)
    if goal_set.issubset(start):
        return []
    visited = {start}
    queue = deque([(start, [])])
    while queue:
        cur_facts, path = queue.popleft()
        if len(path) >= max_depth:
            continue
        for action in legal_actions_block(cur_facts, blocks):
            verb, a1, a2 = action
            args = {1: a1, 2: a2} if a2 else {1: a1}
            new_facts = frozenset(apply_effects(cur_facts, BLOCK_RULES, verb, args))
            new_path = path + [action]
            if goal_set.issubset(new_facts):
                return new_path
            if new_facts not in visited:
                visited.add(new_facts)
                queue.append((new_facts, new_path))
    return None


def validate_and_complete(facts, goals, objects, llm_actions, rules, bfs_fn, max_depth=14):
    cur_facts = set(facts)
    validated = []
    seen = {frozenset(cur_facts)}
    for verb, a1, a2 in llm_actions:
        if verb not in rules:
            break
        if a1 not in objects or (a2 is not None and a2 not in objects):
            break
        args = {1: a1, 2: a2} if a2 else {1: a1}
        if not check_pre(cur_facts, rules, verb, args):
            break
        new_facts = apply_effects(cur_facts, rules, verb, args)
        new_state = frozenset(new_facts)
        if new_state in seen:
            break
        cur_facts = new_facts
        seen.add(new_state)
        validated.append((verb, a1, a2))
        if goals.issubset(cur_facts):
            return validated

    if goals.issubset(cur_facts):
        return validated

    remaining_depth = max(max_depth - len(validated), 0)
    completion = bfs_fn(cur_facts, goals, objects, max_depth=remaining_depth)
    if completion is None:
        return validated
    return validated + completion


def parsed_plan_to_action_tuples(action_strs):
    out = []
    for s in action_strs:
        inner = s.strip("()")
        parts = inner.split()
        if not parts:
            continue
        verb = parts[0]
        a1 = parts[1] if len(parts) > 1 else None
        a2 = parts[2] if len(parts) > 2 else None
        out.append((verb, a1, a2))
    return out

In [ ]:
HYBRID_PLAN_FEAST_SYSTEM = (
    "You are a STRIPS planner. Facts you track: harmony (yes/no), and for "
    "each object: province(yes/no), planet(yes/no), pain(yes/no), craves(target/none).\n"
    "Rules:\n"
    "attack X: needs province(X)=yes,planet(X)=yes,harmony=yes -> pain(X)=yes; province(X)=planet(X)=harmony=no\n"
    "succumb X: needs pain(X)=yes -> province(X)=planet(X)=harmony=yes; pain(X)=no\n"
    "overcome X from Y: needs province(Y)=yes,pain(X)=yes -> harmony=yes,province(X)=yes,craves(X)=Y; province(Y)=no,pain(X)=no\n"
    "feast X from Y: needs craves(X)=Y,province(X)=yes,harmony=yes -> pain(X)=yes,province(Y)=yes; craves(X)=none,province(X)=no,harmony=no\n"
    "Think step by step about which facts change, then output ONLY the final "
    "plan as raw action lines, one per line, exact phrasing "
    "('attack object a', 'feast object b from object c', 'succumb object a', "
    "'overcome object b from object c'), ending with [PLAN END]. "
    "Prefer the shortest correct plan."
)

HYBRID_PLAN_BLOCK_SYSTEM = (
    "You are a STRIPS planner for blocks. Facts you track per block: "
    "ontable(yes/no), clear(yes/no), holding(yes/no), on(target/none). "
    "handempty(yes/no) is global.\n"
    "Rules:\n"
    "pick up X: needs ontable(X)=yes,clear(X)=yes,handempty=yes -> holding(X)=yes; ontable(X)=clear(X)=handempty=no\n"
    "put down X: needs holding(X)=yes -> ontable(X)=clear(X)=handempty=yes; holding(X)=no\n"
    "stack X on Y: needs holding(X)=yes,clear(Y)=yes -> on(X)=Y,clear(X)=yes,handempty=yes; holding(X)=clear(Y)=no\n"
    "unstack X from Y: needs on(X)=Y,clear(X)=yes,handempty=yes -> holding(X)=yes,clear(Y)=yes; on(X)=none,clear(X)=handempty=no\n"
    "Think step by step about which facts change, then output ONLY the final "
    "plan as raw action lines, one per line, exact phrasing "
    "('pick up the red block', 'put down the red block', "
    "'stack the red block on top of the blue block', "
    "'unstack the red block from on top of the blue block'), ending with [PLAN END]. "
    "Prefer the shortest correct plan."
)


def solve_hybrid_batch(scenario_contexts: list, llm_engine_batch_func, max_steps: int = 12) -> list:
    feast_idx, block_idx = [], []
    parsed_states = {}

    for i, ctx in enumerate(scenario_contexts):
        is_block = is_block_domain(ctx)
        if is_block:
            facts, goals, objects = parse_block_state(ctx)
            block_idx.append(i)
        else:
            facts, goals = parse_feast_state(ctx)
            objects = set()
            for f in facts:
                objects.update(f[1:])
            for g in goals:
                objects.update(g[1:])
            feast_idx.append(i)
        parsed_states[i] = (facts, goals, objects, is_block)

    results = [None] * len(scenario_contexts)

    if feast_idx:
        prompts = [scenario_contexts[i] + "\n" for i in feast_idx]
        outs = llm_engine_batch_func(
            prompts, system=HYBRID_PLAN_FEAST_SYSTEM, max_new_tokens=170
        )
        for i, out in zip(feast_idx, outs):
            action_strs = parse_plan(out, is_block=False)
            llm_actions = parsed_plan_to_action_tuples(action_strs)
            facts, goals, objects, _ = parsed_states[i]
            final_actions = validate_and_complete(
                facts, goals, objects, llm_actions, FEAST_RULES, bfs_plan_feast,
                max_depth=18,
            )
            results[i] = [action_to_tuple_str(a) for a in final_actions]

    if block_idx:
        prompts = [scenario_contexts[i] + "\n" for i in block_idx]
        outs = llm_engine_batch_func(
            prompts, system=HYBRID_PLAN_BLOCK_SYSTEM, max_new_tokens=240
        )
        for i, out in zip(block_idx, outs):
            action_strs = parse_plan(out, is_block=True)
            llm_actions = parsed_plan_to_action_tuples(action_strs)
            facts, goals, objects, _ = parsed_states[i]
            final_actions = validate_and_complete(
                facts, goals, objects, llm_actions, BLOCK_RULES, bfs_plan_block,
                max_depth=18,
            )
            results[i] = [action_to_tuple_str(a) for a in final_actions]

    return results

In [ ]:
def limpiar_accion(accion_texto):
    texto = accion_texto.replace('(', '').replace(')', '')
    return texto.strip().lower()


def calcular_score_plan(plan_generado, plan_optimo):
    P = [limpiar_accion(p) for p in plan_generado if p.strip()]
    G = [limpiar_accion(p) for p in plan_optimo if p.strip()]
    L_P = len(P)
    L_G = len(G)
    if L_P == 0:
        return 0.0
    score_horizonte = 2.0 if L_P == L_G else 0.0
    l_match = 0
    for p_accion, g_accion in zip(P, G):
        if p_accion == g_accion:
            l_match += 1
        else:
            break
    score_progreso = 3.0 * (l_match / L_G)
    score_exacto = 5.0 if (l_match == L_G and L_P == L_G) else 0.0
    return round(score_horizonte + score_progreso + score_exacto, 2)


def main(archivo_evaluacion="Task.json", archivo_salida="submission.json", max_steps=12, batch_size=25):
    with open(archivo_evaluacion, "r") as f:
        casos = json.load(f)

    start = time.time()
    contexts = [c["scenario_context"] for c in casos]

    try:
        planes = []
        for i in range(0, len(contexts), batch_size):
            chunk = contexts[i:i + batch_size]
            chunk_planes = solve_hybrid_batch(chunk, qwen_batch, max_steps=max_steps)
            planes.extend(chunk_planes)
            torch.cuda.empty_cache()
            print(f"[batch {i // batch_size + 1}] procesadas {len(planes)}/{len(contexts)} | t={time.time() - start:.1f}s")
    except Exception as e:
        print(f"ERROR critico en batch hybrid: {e}")
        return

    resultados_entrega = []
    for caso, plan_generado in zip(casos, planes):
        resultados_entrega.append({
            "assembly_task_id": caso["assembly_task_id"],
            "complexity_level": len(plan_generado),
            "target_action_sequence": plan_generado,
        })
        print(f"{caso['assembly_task_id']} | acciones={len(plan_generado)}")

    with open(archivo_salida, "w") as f:
        json.dump(resultados_entrega, f, indent=4)

    print("-" * 50)
    print(f"Exito. Archivo '{archivo_salida}' generado correctamente.")
    print(f"Tiempo total: {time.time() - start:.1f}s")

In [ ]:
# Benchmark en Examples.json — ver score antes de entregar
def benchmark_hybrid(n_casos=20, archivo="Examples.json", batch_size=20, max_steps=12):
    with open(archivo, "r") as f:
        casos = json.load(f)
    casos = casos[:n_casos]
    contexts = [c["scenario_context"] for c in casos]
    targets = [c["target_action_sequence"] for c in casos]

    t0 = time.time()
    all_plans = []
    for start in range(0, len(contexts), batch_size):
        chunk = contexts[start:start + batch_size]
        plans = solve_hybrid_batch(chunk, qwen_batch, max_steps=max_steps)
        all_plans.extend(plans)
    t_total = time.time() - t0

    puntaje_total = 0.0
    for i, plan in enumerate(all_plans):
        score = calcular_score_plan(plan, targets[i])
        puntaje_total += score
        print(f"  {casos[i]['assembly_task_id']} | score={score:.2f} | plan={plan}")

    promedio = puntaje_total / len(casos)
    t_promedio = t_total / len(casos)
    print("=" * 60)
    print(f"HYBRID | score={promedio:.2f}/10 | total={t_total:.1f}s | por_tarea={t_promedio:.3f}s")
    proyeccion = t_promedio * 60
    alerta = "  <-- supera 2 min!" if proyeccion > 120 else ""
    print(f"Proyeccion 60 tareas: ~{proyeccion:.0f}s{alerta}")
    return {"score_promedio": round(promedio, 2), "tiempo_total_s": round(t_total, 1)}


benchmark_hybrid(n_casos=20)

In [ ]:
main(archivo_evaluacion="Task.json", archivo_salida="submission.json", max_steps=12, batch_size=25)